
# Publicación en el `Online Feature Store`

**Autor**: Juana María Rodríguez García

El objetivo de esta libreta es mostrar cómo publicar las tablas de la capa `Gold` en un `Online Feature Store` respaldado por `Lakebase` para inferencia en tiempo real.

Esta libreta **no contiene lógica de transformación propia**. Es un *script* de configuración que registra las tablas en el `Online Feature Store` y, la primera vez, lanza la sincronización inicial completa. Las ejecuciones posteriores son incrementales: solo propagan los cambios acumulados desde la última sincronización.

La arquitectura que publicaríamos es la siguiente:

* **`gold_fraud_spine`**: esqueleto de entrenamiento. **No se publica** aquí porque no es una tabla de características: es el andamio que el *job* de entrenamiento usa para buscar las características. Nunca se consulta durante la inferencia.
* **`gold_policy_profile`**: reconocida automáticamente como tabla de características por su clave primaria `(policy_id, __START_AT TIMESERIES)`. Se publicaría en el `Online Feature Store` para que la capa de *serving* recupere el perfil demográfico actual del cliente en tiempo real.
* **`gold_policy_aggregations`**: reconocida automáticamente como tabla de características por su clave primaria `(policy_id, timestamp TIMESERIES)`. Se publicaría en el `Online Feature Store` para consultas de señales de comportamiento durante la inferencia.


## 1. Importación de librerías y configuración

La primera celda instala el paquete `databricks-feature-engineering`, que debe ejecutarse al inicio de cada sesión del clúster. A continuación se importa el cliente del `Feature Store` (`FeatureEngineeringClient`) y se definen los nombres completamente cualificados de las tablas sobre las que vamos a operar. En `Unity Catalog`, un nombre completamente cualificado sigue el formato `catálogo.esquema.tabla`.

El cliente utiliza automáticamente las credenciales del clúster activo, por lo que no es necesario gestionar claves de `API` de forma manual.

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
catalog = "workspace"
database = "fraude-seguros-automovil"   

# Tablas de origen 
gold_policy_profile_table = f"{catalog}.{database}.gold_policy_profile"
gold_policy_aggregations_table = f"{catalog}.{database}.gold_policy_aggregations"

# Nombre del almacén online 
online_store_name = "fraude_seguros_automoviles_online_store"

# Nombres de las tablas dentro del Online Store
online_profile_table = f"{catalog}.{database}.online_policy_profile"
online_aggregations_table = f"{catalog}.{database}.online_policy_aggregations"

# Instanciar el cliente
fe = FeatureEngineeringClient()


## 2. Creación del `Online Feature Store`

El `Online Feature Store` es una instancia gestionada de `Lakebase` (base de datos `PostgreSQL` administrada por `Databricks`) que sirve características con **latencia inferior a 10 milisegundos**. Es la pieza que permite que el modelo de *serving* recupere el perfil y el comportamiento de un cliente en tiempo real sin tener que consultar las tablas `Delta`, cuyas lecturas pueden tardar varios segundos.

Un único `Online Feature Store` puede alojar múltiples tablas de características, por lo que crearíamos uno solo para todo el proyecto. Las opciones de capacidad (`CU_1`, `CU_2`, `CU_4`, `CU_8`) corresponden a distintos niveles de rendimiento: cada unidad de capacidad asigna aproximadamente 16 GB de RAM a la instancia.

El flujo en producción sería el siguiente:

1. El *pipeline* `Gold` se ejecuta en batch cada hora y escribe las nuevas agregaciones en las tablas `Delta`.
2. Al finalizar el *pipeline* `Gold`, este *job* de publicación se lanza automáticamente y propaga de forma incremental los cambios al `Online Feature Store`.
3. Cuando llega una nueva transacción, el modelo de *serving* consulta el `Online Feature Store` por `policy_id` y obtiene las características en menos de 10 milisegundos.
4. El modelo predice si la transacción es fraudulenta y devuelve el resultado.

> **Código comentado**: `Lakebase` no está habilitado en la `Free Edition`. El siguiente bloque muestra cómo se crearía el `Online Feature Store` en un entorno con licencia completa.

In [0]:
capacity = "CU_1"  # Capacity of the online store

# Create the online store only if it does not exist yet.
# On scheduled runs, the store already exists and we just retrieve it.
# try:
#     fe.create_online_store(
#         name = online_store_name,
#         capacity = capacity
#     )
#     print(f"Online store created: {online_store_name}")
# except Exception as e:
#     if "already exists" in str(e).lower():
#         print(f"Online store already exists, skipping creation: {online_store_name}")
#     else:
#         raise

# Retrieve the online store instance to use in the publish step below.
# We must wait until its state is AVAILABLE before publishing.
# online_store = fe.get_online_store(name = online_store_name)
# print(f"Online store state: {online_store.state}")


## 3. Publicación de las tablas en el `Online Feature Store`

La publicación sincroniza los datos desde las tablas `Delta` *offline* hacia el `Online Feature Store`. Para que esta publicación pueda propagar los cambios de forma **incremental**, el `Online Feature Store` requiere que el `Change Data Feed` esté habilitado en las tablas de origen. En nuestro caso ya lo está, porque lo declaramos en los `table_properties` de los *scripts* del *pipeline* `Gold`.

En este *script* utilizamos el modo de publicación `TRIGGERED`, cuya elección está directamente ligada al diseño del *pipeline* `Gold`. Como se puede ver en el *script* correspondiente, la tabla `gold_policy_aggregations` se construye con una **ventana deslizante** (*rolling window*). A diferencia de una ventana fija (*tumbling window*), la ventana deslizante no permite procesamiento incremental: cada nueva transacción obliga a recalcular todas las métricas del cliente desde cero. Ejecutar ese recálculo completo con cada transacción individual tendría un coste computacional desproporcionado.

Por ese motivo, el *pipeline* `Gold` se ejecuta en modo ***batch***, acumulando todas las transacciones del intervalo y recalculando las agregaciones una sola vez por ciclo. Este diseño hace que el modo `CONTINUOUS` en el `Online Feature Store` sea innecesario: no tendría sentido mantener un proceso de sincronización permanente observando cambios que solo se producen por lotes. El modo `TRIGGERED` es la elección natural: propaga exactamente los cambios acumulados desde la última sincronización y libera todos los recursos de cómputo entre ejecuciones.

Una vez publicadas, **el modelo recupera las características automáticamente** durante la inferencia: la capa de *serving* consulta el `Online Feature Store` por `policy_id` y obtiene todas las características de la póliza en un único acceso de baja latencia, sin que el código del modelo tenga que hacer ninguna `JOIN` ni consulta adicional. Esto lo veremos en la siguiente libreta.

> Si en el futuro se sustituyera la ventana deslizante por una **ventana fija** (*tumbling window*), `gold_policy_aggregations` podría construirse como una *streaming table* con procesamiento verdaderamente incremental. En ese caso, el modo `CONTINUOUS` sería la opción natural, ya que el `Online Feature Store` podría sincronizarse en cuanto llegaran nuevos datos sin necesidad de reprocesar el histórico completo.

> **Código comentado**: el siguiente bloque muestra cómo se publicarían las tablas en un entorno con `Lakebase` habilitado.

In [0]:
publish_mode = "TRIGGERED"  # Controls how the online store stays in sync with the offline tables

# Publish the policy profile table to the online store
# fe.publish_table(
#     online_store = online_store_name,
#     source_table_name = gold_policy_profile_table,
#     online_table_name = online_profile_table,
#     publish_mode = publish_mode
# )
# print(f"Published: {gold_policy_profile_table} to {online_profile_table}")

# Publish the behavioral aggregations table to the online store
# fe.publish_table(
#     online_store = online_store_name,
#     source_table_name = gold_policy_aggregations_table,
#     online_table_name = online_aggregations_table,
#     publish_mode = publish_mode
# )
# print(f"Published: {gold_policy_aggregations_table} to {online_aggregations_table}")


## 4. Conclusiones y siguientes pasos

### ¿Qué hemos visto?

En esta libreta hemos descrito la arquitectura de publicación en el `Online Feature Store`:

1. Las tablas `Delta` de la capa `Gold` son reconocidas automáticamente por el `Feature Store` gracias al `CONSTRAINT` de clave primaria declarado en el *pipeline*.
2. En un entorno con licencia completa, `fe.create_online_store` crearía una instancia `PostgreSQL` gestionada por `Lakebase` con latencia de *serving* inferior a 10 milisegundos.
3. `fe.publish_table` en modo `TRIGGERED` propagaría incrementalmente los cambios de cada ciclo horario del *pipeline* `Gold`, sin mantener recursos activos entre ejecuciones. Esta elección es consecuencia directa del uso de ventana deslizante en `gold_policy_aggregations`: si se usara ventana fija, el procesamiento sería verdaderamente incremental y el modo `CONTINUOUS` sería preferible.
4. Durante la inferencia, el modelo recuperaría las características automáticamente a través de la `API` del `Feature Store`, sin necesidad de hacer `JOIN` explícitos en el código.

### ¿Cuándo volver a ejecutar esta libreta?

* Primera configuración del entorno (cuando `Lakebase` esté disponible): lanza la sincronización inicial completa.
* **Cada hora, de forma programada** (tras la ejecución del *pipeline* `Gold`): propaga los cambios incrementales del último ciclo.
* Cuando se recrea o migra el `Online Feature Store`.
* Cuando se cambia el `publish_mode` o la `capacity`.

### ¿Qué sigue?

Hay dos líneas de trabajo que pueden avanzar en paralelo a partir de este punto.

1. **Orquestación en producción**: para que el `Online Feature Store` se mantenga actualizado de forma autónoma, el siguiente paso sería crear un *job* de `Lakeflow` con cadencia horaria compuesto por dos tareas encadenadas: primero la ejecución del *pipeline* `Gold` (que recalcula `gold_policy_aggregations` y actualiza `gold_policy_profile`), y a continuación la ejecución de esta misma libreta (que propaga los cambios al `Online Feature Store` en modo `TRIGGERED`). Con ese *job* activo, el ciclo completo (desde la llegada de nuevas transacciones hasta su disponibilidad en el `Online Feature Store`) sería completamente automático y no requeriría ninguna intervención manual.

2. **Construcción del conjunto de entrenamiento**: en paralelo, en la siguiente libreta construiremos el conjunto de datos de entrenamiento mediante la `API` `create_training_set`, que realiza automáticamente los joins `PiT` entre la *spine* (`gold_fraud_spine`) y las dos tablas de características (`gold_policy_profile` y `gold_policy_aggregations`) leyendo directamente desde las tablas `Delta` de la capa `Gold`. El join `PiT` se articula a través de objetos `FeatureLookup` con el parámetro `timestamp_lookup_key`, que indica contra qué columna de la *spine* se evalúa el `AS OF`, garantizando que cada fila de entrenamiento recibe únicamente los valores de características que habrían estado disponibles en el momento de la transacción. El `Online Feature Store` no interviene en el entrenamiento: su único propósito es servir características con baja latencia durante la inferencia en tiempo real.